# Estudo de caso: 99% de acerto com fumaça?

Um dataset público de detecção de incêndio, com **os mesmos sensores da caixa de fumaça** do
professor: partícula, gás, calor e umidade, uma leitura por segundo.

- **Dataset:** [Smoke Detection Dataset, no Kaggle](https://www.kaggle.com/datasets/deepcontractor/smoke-detection-dataset)
- **Projeto que gravou os dados:** [Real-time Smoke Detection with AI-based Sensor Fusion, no Hackster](https://www.hackster.io/stefanblattmann/real-time-smoke-detection-with-ai-based-sensor-fusion-1086e6)
  (Stefan Blattmann, 2022)

Quem gravou relatou **99,6% de acerto**. Um artigo científico com o mesmo dataset relatou
**99,8%**. Neste notebook você treina um modelo, chega perto disso e depois descobre de onde vem
o número.

**Como usar no Colab:**
1. Arquivo → Fazer upload de notebook → escolha este arquivo.
2. Rode as células na ordem (Shift + Enter). Na primeira célula de código, escolha o CSV do
   Kaggle (`dataset_detector_fumaca_kaggle.csv`). O arquivo original do Kaggle se chama
   `smoke_detection_iot.csv`: é o mesmo.

## 1) As colunas são os sensores da caixa

O projeto usou uma Arduino Nicla Sense ME com sensores externos: **SPS30** (partícula), **SGP30**
(gás), **SHT31** (calor e umidade) e **BMP388** (pressão). Gravou em vários cenários: ambiente
interno e externo normais, fogo de lenha e de gás num centro de treinamento de bombeiros,
churrasqueira (lenha, carvão e gás) e umidade alta ao ar livre.

| Coluna do Kaggle | O que é | Sensor | Na caixa do professor |
|---|---|---|---|
| `Temperature[C]` | calor (°C) | SHT31 | `temp` |
| `Humidity[%]` | umidade (%) | SHT31 | `umid` |
| `TVOC[ppb]` | gás total | SGP30 | `tvoc` |
| `eCO2[ppm]` | CO2 estimado a partir do gás | SGP30 | `eco2` |
| `Raw H2`, `Raw Ethanol` | hidrogênio e etanol, valores crus | SGP30 | `raw_h2`, `raw_ethanol` |
| `Pressure[hPa]` | pressão atmosférica | BMP388 | `pressao` |
| `PM1.0`, `PM2.5` | massa de partícula | SPS30 | `pm1_0`, `pm2_5` |
| `NC0.5`, `NC1.0`, `NC2.5` | contagem de partículas | SPS30 | `nc0_5`, `nc1_0`, `nc2_5` |
| `UTC` | horário da leitura (segundos) | — | — |
| `CNT` | contador de leituras | — | `amostra` |
| `Fire Alarm` | **o rótulo:** 1 = há fogo | quem gravou | `estimulo` (fumaça ou chama = incêndio) |

In [ ]:
import os
import pandas as pd

ARQUIVO = "dataset_detector_fumaca_kaggle.csv"   # o CSV do Kaggle, com o nome da aula

if not os.path.exists(ARQUIVO):
    from google.colab import files
    print("Nao achei", ARQUIVO, "- escolha o CSV do Kaggle no seu computador:")
    enviados = files.upload()
    ARQUIVO = list(enviados)[0]

df = pd.read_csv(ARQUIVO)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")   # a 1a coluna e so o numero da linha
print(len(df), "linhas,", len(df.columns), "colunas")
df.head()

## 2) Quanto do dataset é fogo

In [ ]:
ROTULO = "Fire Alarm"

fogo = df[ROTULO].mean()
print(f"Fire Alarm = 1 em {fogo:.0%} das linhas")
print(f"Um modelo que sempre responde 'fogo' ja acerta {fogo:.0%}.")

## 3) O treino "de livro"

- 70% das linhas, **sorteadas**, para treinar; 30% para testar.
- Saem o `UTC` (horário) e o `CNT` (contador): não são sensores.
- Ficam os 12 sensores.
- O modelo é uma **árvore de decisão com 3 níveis**: dá para ler a regra que ela aprendeu.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

SENSORES = ["Temperature[C]", "Humidity[%]", "TVOC[ppb]", "eCO2[ppm]", "Raw H2", "Raw Ethanol",
            "Pressure[hPa]", "PM1.0", "PM2.5", "NC0.5", "NC1.0", "NC2.5"]

treino, teste = train_test_split(df, test_size=0.3, random_state=42)

arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore.fit(treino[SENSORES], treino[ROTULO])

acerto_aleatorio = accuracy_score(teste[ROTULO], arvore.predict(teste[SENSORES]))
print(f"Acerto no teste (split aleatorio): {acerto_aleatorio:.3f}")

### A regra que a árvore aprendeu

Cada linha é uma pergunta. Leia de cima para baixo: `class: 1` é fogo, `class: 0` não é.

In [ ]:
print(export_text(arvore, feature_names=SENSORES))

Qual é a **primeira pergunta** da árvore? Ela pergunta a **pressão atmosférica**. A pressão muda
com o tempo do dia (o clima), não com o fogo. Um detector de incêndio que decide pela previsão do
tempo?

Guarde a nota do teste. Antes de confiar nela, vamos ver o que um modelo consegue com **uma
coluna só**.

## 4) Uma coluna só, até o relógio

O mesmo treino e o mesmo teste, com uma coluna por vez.

In [ ]:
COLUNAS_SOZINHAS = ["UTC", "Pressure[hPa]", "TVOC[ppb]", "NC0.5"]

for coluna in COLUNAS_SOZINHAS:
    so_uma = DecisionTreeClassifier(max_depth=3, random_state=42)
    so_uma.fit(treino[[coluna]], treino[ROTULO])
    acerto = accuracy_score(teste[ROTULO], so_uma.predict(teste[[coluna]]))
    print(f"so {coluna:15s} acerto {acerto:.3f}")

O `UTC` é o relógio. Ele não sente fumaça: só sabe **quando** a linha foi gravada. E, sozinho,
acerta **mais** que o sensor de gás sozinho e que o de partícula sozinho.

Se o relógio acerta tanto, o fogo e o não-fogo foram gravados em momentos diferentes. Então o
modelo pode estar reconhecendo **a gravação**, não o fogo. A pressão faz o mesmo: cada dia tem a
sua.

## 5) As gravações

O `CNT` conta as leituras e volta a zero quando começa uma gravação nova.

In [ ]:
df["gravacao"] = (df["CNT"] == 0).cumsum()   # +1 cada vez que o contador volta a zero

resumo = df.groupby("gravacao").agg(linhas=("CNT", "size"),
                                    fogo=(ROTULO, "mean"),
                                    pressao=("Pressure[hPa]", "mean"))
resumo["minutos"] = (resumo["linhas"] / 60).round()
resumo["fogo"] = (resumo["fogo"] * 100).round().astype(int).astype(str) + "%"
resumo["pressao"] = resumo["pressao"].round(1)
resumo

### O rótulo no tempo

Em cada gravação: a linha é o gás (`TVOC`); o fundo vermelho é onde o rótulo diz **fogo**.

In [ ]:
import matplotlib.pyplot as plt

gravacoes = sorted(df["gravacao"].unique())
fig, axs = plt.subplots(len(gravacoes), 1, figsize=(12, 11))

for ax, g in zip(axs, gravacoes):
    d = df[df["gravacao"] == g]
    minutos = d["CNT"] / 60
    ax.fill_between(minutos, 0, 1, where=(d[ROTULO] == 1), color="#ef4444", alpha=0.25,
                    transform=ax.get_xaxis_transform(), label="rotulo = fogo")
    ax.plot(minutos, d["TVOC[ppb]"], color="#333333", linewidth=0.7, label="gas (TVOC)")
    ax.set_title(f"gravacao {g}: {len(d)} linhas, pressao {d['Pressure[hPa]'].mean():.1f} hPa",
                 loc="left", fontsize=10)
    ax.set_ylabel("TVOC (ppb)")

axs[0].legend(loc="upper right")
axs[-1].set_xlabel("minutos desde o inicio da gravacao")
plt.tight_layout()
plt.show()

Repare:

- em cada gravação, o rótulo é **um bloco só**. A gravação 1 tem 53 min sem fogo e depois 6 h
  seguidas com fogo; a 4 é quase toda sem fogo;
- o rótulo **não acompanha o gás**. Na gravação 4, o gás chega ao máximo do sensor (60.000 ppb) e
  o rótulo continua "sem fogo". Na 3, o rótulo diz "fogo" o tempo todo, mesmo quando o gás volta
  a zero. Pode até haver motivo (gás sem fogo existe), mas o rótulo é um só para a gravação
  inteira, não para o que acontece em cada segundo;
- cada gravação tem a sua pressão;
- a 1 e a 2 parecem iguais. A 4 e a 5 também.

## 6) Duas gravações repetidas

Coluna por coluna, a gravação 1 é igual à 2? E a 4 à 5?

In [ ]:
def compara(a, b):
    ga = df[df["gravacao"] == a].reset_index(drop=True)
    gb = df[df["gravacao"] == b].reset_index(drop=True)
    print(f"gravacao {a} x gravacao {b}:")
    for coluna in SENSORES + ["UTC", "CNT", ROTULO]:
        if ga[coluna].equals(gb[coluna]):
            print(f"   {coluna:15s} igual")
        else:
            print(f"   {coluna:15s} DIFERENTE")

compara(1, 2)
compara(4, 5)

Gás, partícula, umidade, pressão e rótulo são **idênticos**, linha por linha. Só mudam a
temperatura e o horário. É a mesma gravação duas vezes.

Então são **3 gravações independentes**: A (a 1 e a 2), B (a 3) e C (a 4 e a 5). No split
aleatório, uma linha do teste pode ter a gêmea no treino: o teste vira prova com gabarito.

In [ ]:
df["independente"] = df["gravacao"].map({1: "A", 2: "A", 3: "B", 4: "C", 5: "C"})
df.groupby("independente").agg(linhas=("CNT", "size"), fogo=(ROTULO, "mean")).round(2)

## 7) O teste que vale: uma gravação inteira fora do treino

Treina com duas gravações e testa na terceira, que o modelo nunca viu. É o que acontece quando o
detector é instalado: outro lugar, outro dia.

In [ ]:
acertos = {}
for fora in ["A", "B", "C"]:
    treino_g = df[df["independente"] != fora]
    teste_g = df[df["independente"] == fora]

    modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
    modelo.fit(treino_g[SENSORES], treino_g[ROTULO])
    acertos[fora] = accuracy_score(teste_g[ROTULO], modelo.predict(teste_g[SENSORES]))

    print(f"fora a gravacao {fora} ({len(teste_g)} linhas, {teste_g[ROTULO].mean():.0%} fogo): "
          f"acerto {acertos[fora]:.3f}")

In [ ]:
nomes = ["split aleatorio", "fora A", "fora B", "fora C"]
valores = [acerto_aleatorio, acertos["A"], acertos["B"], acertos["C"]]
cores = ["#22c55e", "#ef4444", "#ef4444", "#ef4444"]

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.bar(nomes, valores, color=cores)
ax.bar_label(barras, fmt="%.2f")
ax.set_ylim(0, 1.1)
ax.set_ylabel("acerto no teste")
ax.set_title("A mesma arvore, dois jeitos de separar treino e teste")
plt.show()

De quase 99% para perto de zero. Nas gravações B e C, o modelo responde **o contrário** do
certo: ele não aprendeu "fogo", aprendeu a reconhecer **de qual gravação vinha a linha**.

### E se tirar a pressão?

In [ ]:
SEM_PRESSAO = []
for coluna in SENSORES:
    if coluna != "Pressure[hPa]":
        SEM_PRESSAO.append(coluna)

for fora in ["A", "B", "C"]:
    treino_g = df[df["independente"] != fora]
    teste_g = df[df["independente"] == fora]

    modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
    modelo.fit(treino_g[SEM_PRESSAO], treino_g[ROTULO])
    acerto = accuracy_score(teste_g[ROTULO], modelo.predict(teste_g[SEM_PRESSAO]))
    print(f"sem pressao, fora a gravacao {fora}: acerto {acerto:.3f}")

Não melhora: continua perto de zero. O problema não é uma coluna, porque outras também mudam de
uma gravação para outra (umidade, temperatura, a linha de base do gás). O problema é **o jeito
de separar** treino e teste.

## 8) Por que o split aleatório engana

**1. Vizinhos quase iguais.** Uma leitura por segundo: um segundo e o seguinte quase não mudam.

In [ ]:
df.loc[54000:54004, SENSORES + [ROTULO]]   # 5 segundos seguidos da gravacao 4

In [ ]:
no_treino = set(treino.index)
com_vizinha = 0
for i in teste.index:
    if i - 1 in no_treino:
        com_vizinha = com_vizinha + 1

print(f"{com_vizinha / len(teste):.0%} das linhas do teste tem a leitura do segundo anterior no treino")

No split aleatório, o modelo quase sempre já viu a vizinha da linha que vai responder.

**2. O rótulo acompanha a gravação.** Cada gravação é quase toda fogo ou quase toda sem fogo
(seção 5), e duas foram repetidas (seção 6). Quem reconhece a gravação acerta o rótulo sem
entender nada de fogo.

## 9) O que levar

- Leituras seguidas (uma por segundo) **não são independentes**. Separe treino e teste por
  **gravação, rodada ou dia**, nunca por linha sorteada.
- **Desconfie de 99%.** A primeira pergunta é: como foi feito o split?
- O projeto original treinou sem teste separado (*"Holdout Validation is disabled"*, diz o
  Hackster): o 99,6% é o acerto nos próprios dados do treino. O artigo usou o split aleatório.
  Os dois números não dizem como o modelo se sai num lugar novo.
- Na caixa de fumaça do professor, o teste que vale é **treinar numa sessão e testar na outra**,
  gravada em outro dia. No MPU, cada rodada do botão é um trecho: separe **por rodada**.